## 로깅

In [1]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
print(ROOT)

os.chdir(ROOT)  # 모든 상대경로는 이 폴더가 기준이 된다.

c:\hw\hw\python_basic\hanhwa-agent


In [2]:
import logging

# 로그 기본 설정
#   로그 포맷 패턴
#   %(asctime)s : 시간
#   %(levelname)s : 로그레벨
#   %(name)s : 로거이름 
#   %(message)s : 메세지
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    datefmt="%H:%M:%S",  # 시분초,
    force=True
)

# 이름을 붙여 로거 하나를 가져옴
log = logging.getLogger("demo")
log.debug("디버그 모드의 debug 입니다.")
log.info("info 레벨의 로그입니다.")
log.warning("warning 레벨 로그입니다.")
log.error("에러 로그입니다.")

09:45:30 DEBUG    demo: 디버그 모드의 debug 입니다.
09:45:30 INFO     demo: info 레벨의 로그입니다.
09:45:30 WARNING  demo: warning 레벨 로그입니다.
09:45:30 ERROR    demo: 에러 로그입니다.


In [3]:
import sys, logging


# 앞 셀에서 루트 로거를 건드렸다면, 아래와 같이 깨끗이하고 새로 얹는다.
logging.getLogger.handlers = []
for m in [k for k in list(sys.modules) if k.startswith("app.core.logging")]:
    del sys.modules[m]


from backend.app.core.logging import get_logger

logger = get_logger("app.services.document")    # 임시 패키지명
logger.info("문서 적재 완료: %s", "DOC-HR-001")
logger.warning("임계치에 근접합니다. 주의하세요.")


09:45:30 INFO     app.services.document: 문서 적재 완료: DOC-HR-001
09:45:30 WARNING  app.services.document: 임계치에 근접합니다. 주의하세요.


In [4]:
class AgentError(Exception):
    status_code, code = 400, "agent_error"
    def __init__(self, message, *, detail=None):
        super().__init__(message)
        self.message, self.detail = message, detail

class NotFound(AgentError):
    status_code, code = 404, "not_found"

class ExternalServiceError(AgentError):
    status_code, code = 502, "external_error"


def fake_parsing(doc_id):
    if doc_id == "DOC-BAD":
        raise ConnectionError("Connection refused: api.llm.ai:443")
    return f"{doc_id}의 본문 텍스트"

def ingest(doc_id):
    logger.info("적재 시작: %s", doc_id)
    try:
        text = fake_parsing(doc_id)
    except ConnectionError as e:
        logger.exception("문서 파싱 실패: %s", doc_id)
        raise ExternalServiceError("문서 변환 서비스에 연결하지 못했습니다.", detail=str(e)) from e

    logger.info("적재 완료: %s", doc_id)
    return text

In [5]:
ingest("DOC-HR-001")

09:45:30 INFO     app.services.document: 적재 시작: DOC-HR-001
09:45:30 INFO     app.services.document: 적재 완료: DOC-HR-001


'DOC-HR-001의 본문 텍스트'

In [6]:
try:
    ingest("DOC-BAD")
except AgentError as e:
    print("Agent Error")
    print(e.status_code)
    print(e.code)
    print(e.message)

09:45:30 INFO     app.services.document: 적재 시작: DOC-BAD
09:45:30 ERROR    app.services.document: 문서 파싱 실패: DOC-BAD
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_24580\3642756407.py", line 22, in ingest
    text = fake_parsing(doc_id)
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_24580\3642756407.py", line 16, in fake_parsing
    raise ConnectionError("Connection refused: api.llm.ai:443")
ConnectionError: Connection refused: api.llm.ai:443
Agent Error
502
external_error
문서 변환 서비스에 연결하지 못했습니다.
